# Game4Loc Fine-tuning — 연남동 데이터 (도로뷰 + 드론뷰 + 위성뷰)

**Sample4Geo 대비 Game4Loc 핵심 차이점**
| 항목 | Sample4Geo | Game4Loc |
|------|-----------|----------|
| 손실함수 | InfoNCE (이진 매칭) | **WeightedInfoNCE** (거리 기반 가중 손실) |
| 학습 쌍 | 양성(positive)만 | **양성 + 준양성(semi-positive)** |
| 가중치 | 없음 | GPS 거리 기반 0~1 |
| 샘플링 | 위성당 쿼리 1개 재선택 | **상호 배제 샘플링** (배치 내 위성 중복 방지) |

**데이터 구조 (Drive)**
```
MyDrive/
├── yeongnamdong_dataset.zip   (또는 이미 압축 해제된 폴더)
├── weights/                   (사전학습 가중치)
└── game4loc_output/           (체크포인트 저장)
```

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 데이터 압축 해제 (Drive → Colab 로컬 SSD)

In [ ]:
import os, time

ZIP_PATH = '/content/drive/MyDrive/yeongnamdong_dataset.zip'
OUT_DIR  = '/content/yeongnamdong_dataset'

if os.path.exists(OUT_DIR):
    print(f'이미 압축 해제됨: {OUT_DIR}')
else:
    print('압축 해제 중... (2~5분 소요)')
    t0 = time.time()
    !unzip -q "{ZIP_PATH}" -d /content/
    print(f'완료! ({time.time()-t0:.0f}초)')

for folder in ['satellite', 'panorama', 'drone']:
    n = len([f for f in os.listdir(f'{OUT_DIR}/{folder}') if f.endswith('.png')])
    print(f'  {folder}: {n}개')

## 3. Game4Loc 클론 및 패키지 설치

In [ ]:
import os

if not os.path.exists('/content/Game4Loc'):
    !git clone https://github.com/Yux1angJi/GTA-UAV.git /content/GTA-UAV
    !cp -r /content/GTA-UAV/Game4Loc /content/Game4Loc
else:
    print('Game4Loc already cloned')

!pip install -q -r /content/Game4Loc/requirements.txt
!pip install -q pytorch_metric_learning
print('설치 완료')

## 4. 공통 임포트

In [ ]:
import sys
import os
import time
import gc
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
from dataclasses import dataclass, field
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler
from scipy.spatial import KDTree
from tqdm import tqdm
from transformers import (
    get_cosine_schedule_with_warmup,
    get_polynomial_decay_schedule_with_warmup,
    get_constant_schedule_with_warmup,
)

sys.path.insert(0, '/content/Game4Loc')
from game4loc.models.model import DesModel
from game4loc.loss import WeightedInfoNCE
from game4loc.trainer.trainer import train_with_weight
from game4loc.utils import setup_system

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 5. 설정 (여기만 수정)

| 항목 | 설명 |
|------|------|
| `n_semipos` | 쿼리당 준양성 위성 수 (0이면 양성만 사용) |
| `semipos_dist_thr` | 준양성 GPS 거리 임계값 (위도·경도 단위) |
| `share_weights` | True: 드론/위성 인코더 가중치 공유 |
| `with_weight` | True: WeightedInfoNCE 사용 |
| `k` | 가중치 곡선 민감도 (클수록 양성/준양성 구분 명확) |

In [ ]:
@dataclass
class Config:

    # ── 경로 ───────────────────────────────────────────────────────────────
    data_folder:      str  = '/content/yeongnamdong_dataset'
    checkpoint_start: str  = '/content/drive/MyDrive/weights/sample4geo_cvusa.pth'
    save_path:        str  = '/content/drive/MyDrive/game4loc_output'

    # ── 학습 쿼리 뷰 선택 ──────────────────────────────────────────────────
    # 'panorama' / 'drone' / ['panorama', 'drone']
    query_types: object = field(default_factory=lambda: ['panorama', 'drone'])

    # ── 모델 ───────────────────────────────────────────────────────────────
    model:         str  = 'convnext_base.fb_in22k_ft_in1k_384'
    model_hub:     str  = 'timm'
    img_size:      int  = 384
    share_weights: bool = True    # 드론/위성 인코더 가중치 공유

    # ── 데이터 분할 ─────────────────────────────────────────────────────────
    val_split: float = 0.2
    seed:      int   = 42

    # ── 준양성(Semi-positive) 설정 ──────────────────────────────────────────
    n_semipos:         int   = 2      # 쿼리당 준양성 위성 수 (0: 양성만)
    semipos_dist_thr:  float = 0.003  # GPS 거리 임계값 (도 단위, ~330m)

    # ── 학습 ───────────────────────────────────────────────────────────────
    mixed_precision: bool  = True
    epochs:          int   = 10
    batch_size:      int   = 16
    verbose:         bool  = True
    gpu_ids:         tuple = (0,)

    # ── Game4Loc 손실 설정 ──────────────────────────────────────────────────
    with_weight:     bool  = True   # WeightedInfoNCE 사용
    label_smoothing: float = 0.1
    k:               float = 3.0    # 가중치 시그모이드 민감도

    # ── 데이터 증강 ─────────────────────────────────────────────────────────
    prob_flip: float = 0.5

    # ── 평가 ───────────────────────────────────────────────────────────────
    batch_size_eval:    int  = 32
    eval_every_n_epoch: int  = 2
    normalize_features: bool = True

    # ── 옵티마이저 ──────────────────────────────────────────────────────────
    lr:                float = 1e-4
    scheduler:         str   = 'cosine'
    warmup_epochs:     int   = 1
    lr_end:            float = 1e-6
    clip_grad:         float = 100.
    decay_exclue_bias: bool  = False
    grad_checkpointing:bool  = False

    # ── 시스템 ─────────────────────────────────────────────────────────────
    device:              str  = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_workers:         int  = 2
    cudnn_benchmark:     bool = True
    cudnn_deterministic: bool = False


config = Config()
os.makedirs(config.save_path, exist_ok=True)

if isinstance(config.query_types, str):
    config.query_types = [config.query_types]

print('학습 쿼리 타입:', config.query_types)
print('데이터 경로:  ', config.data_folder)
print('가중치 경로:  ', config.checkpoint_start)
print('저장 경로:    ', config.save_path)
print('device:       ', config.device)
print(f'준양성 설정:   n_semipos={config.n_semipos}, threshold={config.semipos_dist_thr}')

## 6. 커스텀 Dataset

### 설계 원칙

**양성(Positive) vs 준양성(Semi-positive)**
```
쿼리 A ──────────── 위성 X  (가중치=1.0, GPS 거리 최소) → WeightedInfoNCE: eps≈0.05
                  ↘ 위성 Y  (가중치≈0.5, 2번째 가까운)  → WeightedInfoNCE: eps≈0.18
                  ↘ 위성 Z  (가중치≈0.3, 3번째 가까운)  → WeightedInfoNCE: eps≈0.30
```
WeightedInfoNCE는 가중치가 높을수록 강한 매칭 신호, 낮을수록 부드러운 신호를 부여합니다.

**상호 배제 샘플링(Mutually Exclusive Sampling)**
- epoch마다 `shuffle()` 호출: 배치 내 동일 위성 중복 방지
- 쿼리 A가 위성 X로 선택되면, 위성 Y·Z·X 모두 해당 배치에서 제외
- → False Negative 없는 배치 구성 보장

In [ ]:
def _load_meta(data_folder, view_type):
    df = pd.read_csv(os.path.join(data_folder, view_type, 'meta.csv'))
    return [
        (os.path.join(data_folder, row['file']), float(row['lat']), float(row['lng']))
        for _, row in df.iterrows()
    ]


def _build_pairs(data_folder, query_types, n_semipos, semipos_dist_thr, val_split, seed):
    """
    meta.csv → (query_path, sat_path, weight, sat_idx) 쌍 생성
    - 양성: 최근접 위성, weight=1.0
    - 준양성: 2~n번째 가까운 위성 (threshold 이내), weight=exp(-d/d0)*0.7 클램프
    위성 기준 train/val 분리 (데이터 누수 없음)
    """
    sat_records = _load_meta(data_folder, 'satellite')
    sat_coords  = np.array([(r[1], r[2]) for r in sat_records])
    tree        = KDTree(sat_coords)

    k_query = 1 + n_semipos  # positive + semi-positives
    all_pairs = []

    for qtype in query_types:
        q_records = _load_meta(data_folder, qtype)
        q_coords  = np.array([(r[1], r[2]) for r in q_records])
        dists, idxs = tree.query(q_coords, k=k_query)

        for qi in range(len(q_records)):
            q_path = q_records[qi][0]
            pos_dist = float(dists[qi][0])
            pos_idx  = int(idxs[qi][0])

            # 양성 쌍
            all_pairs.append((q_path, sat_records[pos_idx][0], 1.0, pos_idx))

            # 준양성 쌍
            for kk in range(1, k_query):
                sp_dist = float(dists[qi][kk])
                sp_idx  = int(idxs[qi][kk])
                if sp_dist > semipos_dist_thr:
                    break
                # 양성 거리 대비 상대적 거리로 가중치 계산
                weight = float(np.exp(-sp_dist / max(pos_dist, 1e-8)) * 0.7)
                weight = float(np.clip(weight, 0.1, 0.9))
                all_pairs.append((q_path, sat_records[sp_idx][0], weight, sp_idx))

    # 위성 기준 train/val 분리
    unique_sats = list({p[3] for p in all_pairs})
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_sats)
    n_val    = int(len(unique_sats) * val_split)
    val_sats = set(unique_sats[:n_val])
    trn_sats = set(unique_sats[n_val:])

    train_pairs = [p for p in all_pairs if p[3] in trn_sats]
    val_pairs   = [p for p in all_pairs if p[3] in val_sats]
    return train_pairs, val_pairs, sat_records


def _filter_missing(pairs, tag=''):
    valid = [(q, s, w, si) for q, s, w, si in pairs
             if os.path.exists(q) and os.path.exists(s)]
    n_removed = len(pairs) - len(valid)
    if n_removed:
        print(f'  [경고]{tag} 파일 없음 {n_removed}개 제외')
    return valid


class YeongnamdongG4LTrainDataset(Dataset):
    """
    Game4Loc 학습용 Dataset.

    반환값: (query_img, sat_img, weight)
    - weight: 양성=1.0, 준양성=0.1~0.9 (GPS 거리 기반)
    - WeightedInfoNCE가 weight를 eps(label smoothing 강도)로 변환

    shuffle() → 상호 배제 샘플링
    - 배치 내 동일 쿼리/위성 중복 없음 → False Negative 방지

    transforms_query: 단일 callable 또는 dict {'panorama': tf, 'drone': tf}
    - dict로 넘기면 쿼리 경로에서 뷰 타입을 감지해 적절한 transform 선택
    - 파노라마(street-level): RandomRotate90 없는 transform 권장
    - 드론(top-down): RandomRotate90 포함 transform 사용 가능
    """

    def __init__(self, data_folder, query_types,
                 transforms_query=None, transforms_gallery=None,
                 prob_flip=0.5,
                 n_semipos=2, semipos_dist_thr=0.003,
                 val_split=0.2, seed=42):
        super().__init__()
        train_pairs, _, _ = _build_pairs(
            data_folder, query_types,
            n_semipos, semipos_dist_thr,
            val_split, seed)

        self.pairs = _filter_missing(train_pairs, tag='[train]')

        # 상호 배제 샘플링을 위한 그래프
        self._q2s = {}   # query_name → [sat_names]
        self._s2q = {}   # sat_name  → [query_names]
        for q_path, s_path, w, si in self.pairs:
            qn = os.path.basename(q_path)
            sn = os.path.basename(s_path)
            self._q2s.setdefault(qn, []).append(sn)
            self._s2q.setdefault(sn, []).append(qn)

        self.transforms_query   = transforms_query
        self.transforms_gallery = transforms_gallery
        self.prob_flip          = prob_flip
        self.samples = copy.deepcopy(self.pairs)

        n_pos  = sum(1 for p in self.pairs if p[2] == 1.0)
        n_semi = len(self.pairs) - n_pos
        unique_sats = len({p[3] for p in self.pairs})
        print(f'  전체 쌍: {len(self.pairs)}  '
              f'(양성={n_pos} / 준양성={n_semi})')
        print(f'  학습 위성 수: {unique_sats}')

    def shuffle(self):
        """상호 배제 샘플링: 배치 내 쿼리·위성 중복 없이 선택"""
        pair_pool = copy.deepcopy(self.pairs)
        random.shuffle(pair_pool)

        sate_used  = set()
        query_used = set()
        pairs_used = set()
        selected   = []

        for pair in pair_pool:
            q_path, s_path, w, s_idx = pair
            qn = os.path.basename(q_path)
            sn = os.path.basename(s_path)
            pn = (qn, sn)

            if qn in query_used or sn in sate_used or pn in pairs_used:
                continue

            selected.append(pair)
            pairs_used.add(pn)
            # 이 쿼리와 연결된 모든 위성을 사용 완료로 표시
            for s in self._q2s.get(qn, []):
                sate_used.add(s)
            # 이 위성과 연결된 모든 쿼리를 사용 완료로 표시
            for q in self._s2q.get(sn, []):
                query_used.add(q)

        self.samples = selected
        print(f'셔플 완료: {len(self.pairs)}쌍 → {len(self.samples)}쌍 선택')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        q_path, s_path, weight, sat_idx = self.samples[index]

        q_img = cv2.imread(q_path)
        s_img = cv2.imread(s_path)
        if q_img is None:
            raise FileNotFoundError(f'쿼리 이미지 없음: {q_path}')
        if s_img is None:
            raise FileNotFoundError(f'위성 이미지 없음: {s_path}')
        q_img = cv2.cvtColor(q_img, cv2.COLOR_BGR2RGB)
        s_img = cv2.cvtColor(s_img, cv2.COLOR_BGR2RGB)

        if np.random.random() < self.prob_flip:
            q_img = cv2.flip(q_img, 1)
            s_img = cv2.flip(s_img, 1)

        # dict transforms: 경로에서 뷰 타입 감지 후 적절한 transform 선택
        tf_q = self.transforms_query
        if isinstance(tf_q, dict):
            key = 'panorama' if 'panorama' in q_path else 'drone'
            tf_q = tf_q.get(key, tf_q.get('drone'))
        if tf_q:
            q_img = tf_q(image=q_img)['image']
        if self.transforms_gallery:
            s_img = self.transforms_gallery(image=s_img)['image']

        return q_img, s_img, torch.tensor(weight, dtype=torch.float32)


class YeongnamdongG4LEvalDataset(Dataset):
    """
    평가용 Dataset.
    img_type='query'     → val 쿼리 (양성 쌍만), 레이블=sat_idx
    img_type='reference' → 전체 위성 갤러리,     레이블=sat_idx
    """

    def __init__(self, data_folder, img_type, query_types=None,
                 transforms=None, val_split=0.2, split='val', seed=42,
                 n_semipos=0):
        super().__init__()
        self.transforms = transforms

        if img_type == 'reference':
            sat_records = _load_meta(data_folder, 'satellite')
            images = [r[0] for r in sat_records]
            labels = list(range(len(sat_records)))

        elif img_type == 'query':
            # 양성 쌍만 평가에 사용 (n_semipos=0)
            train_pairs, val_pairs, _ = _build_pairs(
                data_folder, query_types or ['panorama'],
                n_semipos=0, semipos_dist_thr=0,
                val_split=val_split, seed=seed)

            pairs = val_pairs if split == 'val' else train_pairs + val_pairs
            images = [p[0] for p in pairs]
            labels = [p[3] for p in pairs]
        else:
            raise ValueError("img_type must be 'query' or 'reference'")

        # 존재하지 않는 파일 제거
        valid = [(p, l) for p, l in zip(images, labels) if os.path.exists(p)]
        n_removed = len(images) - len(valid)
        if n_removed:
            print(f'  [경고][{img_type}/{split}] 파일 없음 {n_removed}개 제외')
        if valid:
            self.images, self.labels = zip(*valid)
            self.images = list(self.images)
            self.labels = list(self.labels)
        else:
            self.images, self.labels = [], []

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        path = self.images[index]
        img  = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f'이미지 없음: {path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transforms:
            img = self.transforms(image=img)['image']
        return img, torch.tensor(self.labels[index], dtype=torch.long)


print('Dataset 클래스 정의 완료')

## 7. 모델 로드 및 사전학습 가중치 적용

In [ ]:
setup_system(seed=config.seed,
             cudnn_benchmark=config.cudnn_benchmark,
             cudnn_deterministic=config.cudnn_deterministic)

model = DesModel(
    model_name=config.model,
    pretrained=True,
    img_size=config.img_size,
    share_weights=config.share_weights,
)

data_config = model.get_config()
mean, std = data_config['mean'], data_config['std']
image_size = (config.img_size, config.img_size)

print(f'모델:          {config.model}')
print(f'가중치 공유:   {config.share_weights}')
print(f'이미지 크기:   {image_size}')
print(f'mean={mean}  std={std}')

if config.grad_checkpointing:
    model.set_grad_checkpointing(True)

if os.path.isfile(config.checkpoint_start):
    print(f'\n사전학습 가중치 로드: {config.checkpoint_start}')
    state = torch.load(config.checkpoint_start, map_location='cpu')
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f'  누락: {len(missing)}  예상치 못한 키: {len(unexpected)}')
else:
    print(f'[경고] 가중치 파일 없음 → timm ImageNet 가중치만 사용')
    print(f'  경로 확인: {config.checkpoint_start}')

model = model.to(config.device)
print('\n모델 준비 완료')

## 8. 데이터로더 구성

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from game4loc.dataset.gta import get_transforms

# game4loc 원본 transforms (albumentations v2 경고 있지만 동작함)
# sat_rot=True: 위성 이미지 90° 회전 증강
# train_drone_transforms: RandomRotate90(p=1.0) 포함 → 드론(top-down)에만 사용
val_transforms, train_sat_transforms, train_drone_transforms = \
    get_transforms(image_size, mean=mean, std=std, sat_rot=True)

# 파노라마(street-level)용 transform: RandomRotate90 제외
# 거리뷰는 sky/ground 방향이 의미 있으므로 90° 회전 시 매칭 신호 붕괴
train_pano_transforms = A.Compose([
    A.Resize(image_size[0], image_size[1], interpolation=cv2.INTER_LINEAR_EXACT, p=1.0),
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.15, p=0.5),
    A.Normalize(mean, std),
    ToTensorV2(),
])

# ── 학습 ────────────────────────────────────────────────────────────────────
train_dataset = YeongnamdongG4LTrainDataset(
    data_folder=config.data_folder,
    query_types=config.query_types,
    transforms_query={
        'panorama': train_pano_transforms,   # 회전 없음
        'drone':    train_drone_transforms,  # RandomRotate90 포함
    },
    transforms_gallery=train_sat_transforms,
    prob_flip=config.prob_flip,
    n_semipos=config.n_semipos,
    semipos_dist_thr=config.semipos_dist_thr,
    val_split=config.val_split,
    seed=config.seed,
)
train_dataloader = DataLoader(
    train_dataset, batch_size=config.batch_size,
    shuffle=False,   # shuffle()로 직접 관리
    num_workers=config.num_workers,
    pin_memory=True, drop_last=True,
)

# ── 검증 쿼리 ───────────────────────────────────────────────────────────────
query_dataset_val = YeongnamdongG4LEvalDataset(
    data_folder=config.data_folder, img_type='query',
    query_types=config.query_types,
    transforms=val_transforms,
    val_split=config.val_split, split='val', seed=config.seed,
)
query_dataloader_val = DataLoader(
    query_dataset_val, batch_size=config.batch_size_eval,
    shuffle=False, num_workers=config.num_workers, pin_memory=True,
)

# ── 검증 레퍼런스 (전체 위성 갤러리) ────────────────────────────────────────
reference_dataset_val = YeongnamdongG4LEvalDataset(
    data_folder=config.data_folder, img_type='reference',
    transforms=val_transforms,
)
reference_dataloader_val = DataLoader(
    reference_dataset_val, batch_size=config.batch_size_eval,
    shuffle=False, num_workers=config.num_workers, pin_memory=True,
)

print(f'학습 쌍:       {len(train_dataset):>6}  (쿼리 타입: {config.query_types})')
print(f'검증 쿼리:     {len(query_dataset_val):>6}')
print(f'검증 레퍼런스: {len(reference_dataset_val):>6}  (전체 위성)')
print(f'파노라마 transform: 회전 없음 / 드론 transform: RandomRotate90 포함')

## 9. 손실함수 / 옵티마이저 / 스케줄러

**WeightedInfoNCE 동작 원리**
```
eps(i) = 1 - 1 / (1 + exp(-k * weight(i)))

weight=1.0 → eps≈0.05  (강한 매칭 신호, 양성)
weight=0.5 → eps≈0.18  (중간 신호, 준양성)
weight=0.1 → eps≈0.43  (약한 신호, 먼 준양성)
```

In [ ]:
loss_function = WeightedInfoNCE(
    label_smoothing=config.label_smoothing,
    k=config.k,
    device=config.device,
)

scaler = GradScaler(init_scale=2.**10) if config.mixed_precision else None

if config.decay_exclue_bias:
    no_decay = ['bias', 'LayerNorm.bias']
    params   = list(model.named_parameters())
    opt_params = [
        {'params': [p for n, p in params if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in params if     any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    optimizer = torch.optim.AdamW(opt_params, lr=config.lr)
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)

train_steps  = len(train_dataloader) * config.epochs
warmup_steps = len(train_dataloader) * config.warmup_epochs

if config.scheduler == 'cosine':
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_training_steps=train_steps, num_warmup_steps=warmup_steps)
elif config.scheduler == 'polynomial':
    scheduler = get_polynomial_decay_schedule_with_warmup(
        optimizer, num_training_steps=train_steps,
        lr_end=config.lr_end, power=1.5, num_warmup_steps=warmup_steps)
elif config.scheduler == 'constant':
    scheduler = get_constant_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps)
else:
    scheduler = None

print(f'손실함수: WeightedInfoNCE  k={config.k}  label_smoothing={config.label_smoothing}')
print(f'with_weight: {config.with_weight}')
print(f'스케줄러: {config.scheduler}  warmup {config.warmup_epochs} epoch ({warmup_steps} step)')
print(f'전체:    {config.epochs} epoch ({train_steps} step)')

## 10. 평가 함수 (Recall@K)

In [ ]:
def _predict_features(config, model, dataloader):
    """DataLoader에서 이미지 특징 추출 (img, label) → (features, labels)"""
    model.eval()
    features_list = []
    labels_list   = []
    with torch.no_grad():
        bar = tqdm(dataloader, total=len(dataloader))
        for img, labels in bar:
            img = img.to(config.device)
            with torch.cuda.amp.autocast():
                feat = model(img)
                if config.normalize_features:
                    feat = F.normalize(feat, dim=-1)
            features_list.append(feat.to(torch.float32))
            labels_list.append(labels)
        bar.close()
    features = torch.cat(features_list, dim=0)
    labels   = torch.cat(labels_list,   dim=0).to(config.device)
    return features, labels


def run_eval(config, model, query_dl, reference_dl, ranks=[1, 5, 10], step_size=500):
    """Recall@K 평가 (query 레이블 == reference 레이블이면 정답)"""
    print('  Feature 추출 중...')
    ref_features,   ref_labels   = _predict_features(config, model, reference_dl)
    query_features, query_labels = _predict_features(config, model, query_dl)

    print('  Score 계산 중...')
    n_query = query_features.shape[0]
    recall  = {r: 0 for r in ranks}

    for start in range(0, n_query, step_size):
        end  = min(start + step_size, n_query)
        sims = query_features[start:end] @ ref_features.T  # (chunk, n_ref)
        top_indices = torch.argsort(sims, dim=1, descending=True)
        top_labels  = ref_labels[top_indices]
        gt_labels   = query_labels[start:end].unsqueeze(1)
        matches     = (top_labels == gt_labels)             # (chunk, n_ref)

        for r in ranks:
            recall[r] += int(matches[:, :r].any(dim=1).sum().item())

    r1 = recall[1] / n_query * 100
    result_str = '  ' + '  '.join(
        f'R@{r}: {recall[r]/n_query*100:.4f}%' for r in ranks)
    print(result_str)

    del ref_features, ref_labels, query_features, query_labels
    gc.collect()
    return r1


print('run_eval 정의 완료')

## 11. 학습 루프

In [ ]:
best_score = 0.0
run_id     = time.strftime('%Y%m%d_%H%M%S')
ckpt_dir   = os.path.join(config.save_path, run_id)
os.makedirs(ckpt_dir, exist_ok=True)
print(f'체크포인트 저장 경로: {ckpt_dir}')
print(f'WeightedInfoNCE with_weight={config.with_weight}')

for epoch in range(1, config.epochs + 1):
    print(f'\n{"-"*28}[Epoch {epoch}/{config.epochs}]{"-"*28}')

    # 상호 배제 샘플링: epoch마다 쌍 재구성
    train_dataset.shuffle()

    train_loss = train_with_weight(
        config, model,
        dataloader=train_dataloader,
        loss_function=loss_function,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        with_weight=config.with_weight,
    )
    print(f'loss={train_loss:.4f}  lr={optimizer.param_groups[0]["lr"]:.2e}')

    if epoch % config.eval_every_n_epoch == 0 or epoch == config.epochs:
        print(f'{"-"*20}[Evaluate]{"-"*20}')
        r1 = run_eval(config, model,
                      query_dataloader_val, reference_dataloader_val)

        if r1 > best_score:
            best_score = r1
            save_file  = os.path.join(ckpt_dir, f'weights_e{epoch}_r1_{r1:.4f}.pth')
            torch.save(model.state_dict(), save_file)
            print(f'  ★ Best R@1={best_score:.4f} → {save_file}')

final_path = os.path.join(ckpt_dir, 'weights_final.pth')
torch.save(model.state_dict(), final_path)
print(f'\n학습 완료  |  Best R@1: {best_score:.4f}')
print(f'최종 가중치: {final_path}')

## 12. 전체 데이터 최종 평가

In [ ]:
query_dataset_all = YeongnamdongG4LEvalDataset(
    data_folder=config.data_folder, img_type='query',
    query_types=config.query_types,
    transforms=val_transforms,
    val_split=config.val_split, split='all', seed=config.seed,
)
query_dataloader_all = DataLoader(
    query_dataset_all, batch_size=config.batch_size_eval,
    shuffle=False, num_workers=config.num_workers, pin_memory=True,
)

print(f'전체 평가 쿼리 수: {len(query_dataset_all)}')
print('\n[전체 데이터 최종 평가]')
r1_final = run_eval(
    config, model,
    query_dataloader_all, reference_dataloader_val,
)
print(f'최종 R@1: {r1_final:.4f}%')

## 13. 뷰 타입별 개별 평가

In [ ]:
for vtype in config.query_types:
    print(f'\n[{vtype.upper()} 단독 평가]')
    ds = YeongnamdongG4LEvalDataset(
        data_folder=config.data_folder, img_type='query',
        query_types=[vtype],
        transforms=val_transforms,
        val_split=config.val_split, split='all', seed=config.seed,
    )
    dl = DataLoader(ds, batch_size=config.batch_size_eval,
                    shuffle=False, num_workers=config.num_workers, pin_memory=True)
    print(f'  쿼리 수: {len(ds)}')
    run_eval(config, model, dl, reference_dataloader_val)

## 14. Best 체크포인트 로드 및 재평가 (선택)

In [ ]:
BEST_WEIGHT_PATH = ''  # 예: '.../weights_e6_r1_15.0000.pth'

if BEST_WEIGHT_PATH and os.path.isfile(BEST_WEIGHT_PATH):
    print(f'로드: {BEST_WEIGHT_PATH}')
    model.load_state_dict(torch.load(BEST_WEIGHT_PATH, map_location=config.device))
    model.to(config.device)
    print('\n[Best 가중치 전체 평가]')
    run_eval(config, model, query_dataloader_all, reference_dataloader_val)
else:
    print('BEST_WEIGHT_PATH를 입력하세요.')